In [1]:
from pathlib import Path
import sys
import csv
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid")

In [2]:
NOTEBOOKS_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOKS_DIR.parent

DATA_RAW_DIR = PROJECT_ROOT / "DATA" / "RAW"
DATA_PROCESSED_DIR = PROJECT_ROOT / "DATA" / "PROCESSED"
SRC_DIR = PROJECT_ROOT / "SRC"

# Support an optional MATCHES subfolder.
if (DATA_RAW_DIR / "MATCHES").is_dir():
    DATA_RAW_DIR = DATA_RAW_DIR / "MATCHES"

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Raw data:", DATA_RAW_DIR)
print("Processed data:", DATA_PROCESSED_DIR)

assert PROJECT_ROOT.exists(), "Project root does not exist."
assert DATA_RAW_DIR.exists(), "Raw data directory does not exist."
assert DATA_PROCESSED_DIR.exists(), "Could not create processed directory."

print("\nDirectory verification successful.")

Project root: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\INDIAN PREMIER LEAGUE
Raw data: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\INDIAN PREMIER LEAGUE\DATA\RAW
Processed data: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\INDIAN PREMIER LEAGUE\DATA\PROCESSED

Directory verification successful.


In [3]:
all_csv_files = sorted(DATA_RAW_DIR.glob("*.csv"))

delivery_files = [
    file for file in all_csv_files
    if not file.stem.endswith("_info")
]

info_files = [
    file for file in all_csv_files
    if file.stem.endswith("_info")
]

delivery_ids = {file.stem for file in delivery_files}
info_ids = {file.stem.removesuffix("_info") for file in info_files}

missing_info = sorted(delivery_ids - info_ids)
missing_delivery = sorted(info_ids - delivery_ids)

print("RAW DATA SUMMARY")
print("-" * 45)
print("All CSV files:", len(all_csv_files))
print("Delivery files:", len(delivery_files))
print("Info files:", len(info_files))
print("Missing info files:", len(missing_info))
print("Missing delivery files:", len(missing_delivery))

assert len(delivery_files) > 0, "No delivery files found."

if not missing_info and not missing_delivery:
    print("\nAll delivery and info files are paired.")
else:
    print("\nSome files are missing their matching pair.")
    print("Missing info IDs:", missing_info[:10])
    print("Missing delivery IDs:", missing_delivery[:10])

RAW DATA SUMMARY
---------------------------------------------
All CSV files: 2486
Delivery files: 1243
Info files: 1243
Missing info files: 0
Missing delivery files: 0

All delivery and info files are paired.


In [4]:
sample_delivery_path = DATA_RAW_DIR / "1082591.csv"

if not sample_delivery_path.exists():
    sample_delivery_path = delivery_files[0]

sample_deliveries = pd.read_csv(sample_delivery_path)

print("Sample delivery file:", sample_delivery_path.name)
print("Shape:", sample_deliveries.shape)
print("\nColumns:")
print(sample_deliveries.columns.tolist())

display(sample_deliveries.head())
display(sample_deliveries.info())

Sample delivery file: 1082591.csv
Shape: (248, 27)

Columns:
['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3']


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3
0,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.1,0.1,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,TS Mills,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.2,0.2,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,TS Mills,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.3,0.3,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,TS Mills,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.4,0.4,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,TS Mills,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.5,0.5,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,S Dhawan,TS Mills,0,2,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248 entries, 0 to 247
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   match_id                248 non-null    int64  
 1   season                  248 non-null    int64  
 2   start_date              248 non-null    object 
 3   venue                   248 non-null    object 
 4   innings                 248 non-null    int64  
 5   ball                    248 non-null    float64
 6   actual_delivery         248 non-null    float64
 7   batting_team            248 non-null    object 
 8   bowling_team            248 non-null    object 
 9   striker                 248 non-null    object 
 10  non_striker             248 non-null    object 
 11  bowler                  248 non-null    object 
 12  runs_off_bat            248 non-null    int64  
 13  extras                  248 non-null    int64  
 14  wides                   8 non-null      fl

None

In [5]:
def parse_match_info(info_path):
    metadata = defaultdict(list)

    with open(info_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file)

        for row in reader:
            if not row:
                continue

            if row[0] == "version":
                metadata["version"].append(row[1:])
            elif row[0] == "info" and len(row) >= 3:
                metadata[row[1]].append(row[2:])

    def first_value(key, default=None):
        values = metadata.get(key, [])
        if not values or not values[0]:
            return default
        return values[0][0]

    teams = [
        values[0]
        for values in metadata.get("team", [])
        if values
    ]

    return {
        "match_id": first_value("match_id"),
        "season": first_value("season"),
        "date": first_value("date"),
        "event": first_value("event"),
        "match_type": first_value("match_type"),
        "team_1": teams[0] if len(teams) > 0 else None,
        "team_2": teams[1] if len(teams) > 1 else None,
        "venue": first_value("venue"),
        "city": first_value("city"),
        "toss_winner": first_value("toss_winner"),
        "toss_decision": first_value("toss_decision"),
        "player_of_match": first_value("player_of_match"),
        "winner": first_value("winner"),
        "winner_runs": first_value("winner_runs"),
        "winner_wickets": first_value("winner_wickets"),
        "overs": first_value("overs"),
        "info_version": first_value("version"),
    }

In [6]:
sample_info_path = sample_delivery_path.with_name(
    f"{sample_delivery_path.stem}_info.csv"
)

assert sample_info_path.exists(), "Matching info file was not found."

sample_match_info = parse_match_info(sample_info_path)

display(pd.DataFrame([sample_match_info]))

,match_id,season,date,event,match_type,team_1,team_2,venue,city,toss_winner,toss_decision,player_of_match,winner,winner_runs,winner_wickets,overs,info_version
0,1082591,2017,2017/04/05,Indian Premier League,T20,Sunrisers Hyderabad,Royal Challengers Bangalore,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Royal Challengers Bangalore,field,Yuvraj Singh,Sunrisers Hyderabad,35,None,20,2.3.0


In [7]:
print("DATA TYPES")
display(
    sample_deliveries.dtypes.rename("dtype").to_frame()
)

missing_summary = (
    sample_deliveries.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    / len(sample_deliveries)
    * 100
).round(2)

print("MISSING VALUES")
display(
    missing_summary.sort_values(
        "missing_percentage", ascending=False
    )
)

DATA TYPES


,dtype
match_id,int64
season,int64
start_date,object
venue,object
innings,int64
ball,float64
actual_delivery,float64
batting_team,object
bowling_team,object
striker,object


MISSING VALUES


,missing_count,missing_percentage
other_wicket_type,248,100.00
byes,248,100.00
penalty,248,100.00
fielder_3,248,100.00
non_boundary,248,100.00
fielder_2,248,100.00
other_player_dismissed,248,100.00
legbyes,246,99.19
noballs,246,99.19
wides,240,96.77


In [8]:
numeric_columns = sample_deliveries.select_dtypes(
    include=np.number
).columns.tolist()

display(sample_deliveries[numeric_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
match_id,248.0,1.082591e+06,0.000000,1082591.0,1082591.000,1082591.00,1082591.000,1082591.0
season,248.0,2.017000e+03,0.000000,2017.0,2017.000,2017.00,2017.000,2017.0
innings,248.0,1.495968e+00,0.500995,1.0,1.000,1.00,2.000,2.0
ball,248.0,9.729839e+00,5.742356,0.1,4.600,9.60,14.525,19.6
actual_delivery,248.0,9.714113e+00,5.743329,0.1,4.575,9.55,14.525,19.6
runs_off_bat,248.0,1.475806e+00,1.751639,0.0,0.000,1.00,2.000,6.0
extras,248.0,5.241935e-02,0.240769,0.0,0.000,0.00,0.000,2.0
wides,8.0,1.125000e+00,0.353553,1.0,1.000,1.00,1.000,2.0
noballs,2.0,1.000000e+00,0.000000,1.0,1.000,1.00,1.000,1.0
byes,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
sample_score = (
    sample_deliveries.groupby(
        ["innings", "batting_team"],
        dropna=False
    )
    .agg(
        delivery_records=("match_id", "size"),
        batter_runs=("runs_off_bat", "sum"),
        extras=("extras", "sum"),
    )
    .reset_index()
)

sample_score["total_runs"] = (
    sample_score["batter_runs"] + sample_score["extras"]
)

display(sample_score)

,innings,batting_team,delivery_records,batter_runs,extras,total_runs
0,1,Sunrisers Hyderabad,125,200,7,207
1,2,Royal Challengers Bangalore,123,166,6,172


In [10]:
sample_records = []

for delivery_path in delivery_files[:10]:
    info_path = delivery_path.with_name(
        f"{delivery_path.stem}_info.csv"
    )

    if info_path.exists():
        sample_records.append(parse_match_info(info_path))

sample_matches = pd.DataFrame(sample_records)

print("Sample metadata records:", len(sample_matches))
display(sample_matches)

Sample metadata records: 10


,match_id,season,date,event,match_type,team_1,team_2,venue,city,toss_winner,toss_decision,player_of_match,winner,winner_runs,winner_wickets,overs,info_version
0,1082591,2017,2017/04/05,Indian Premier League,T20,Sunrisers Hyderabad,Royal Challengers Bangalore,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Royal Challengers Bangalore,field,Yuvraj Singh,Sunrisers Hyderabad,35,None,20,2.3.0
1,1082592,2017,2017/04/06,Indian Premier League,T20,Rising Pune Supergiant,Mumbai Indians,Maharashtra Cricket Association Stadium,Pune,Rising Pune Supergiant,field,SPD Smith,Rising Pune Supergiant,None,7,20,2.3.0
2,1082593,2017,2017/04/07,Indian Premier League,T20,Gujarat Lions,Kolkata Knight Riders,Saurashtra Cricket Association Stadium,Rajkot,Kolkata Knight Riders,field,CA Lynn,Kolkata Knight Riders,None,10,20,2.3.0
3,1082594,2017,2017/04/08,Indian Premier League,T20,Kings XI Punjab,Rising Pune Supergiant,Holkar Cricket Stadium,Indore,Kings XI Punjab,field,GJ Maxwell,Kings XI Punjab,None,6,20,2.3.0
4,1082595,2017,2017/04/08,Indian Premier League,T20,Royal Challengers Bangalore,Delhi Daredevils,M.Chinnaswamy Stadium,Bengaluru,Royal Challengers Bangalore,bat,KM Jadhav,Royal Challengers Bangalore,15,None,20,2.3.0
5,1082596,2017,2017/04/09,Indian Premier League,T20,Sunrisers Hyderabad,Gujarat Lions,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,field,Rashid Khan,Sunrisers Hyderabad,None,9,20,2.3.0
6,1082597,2017,2017/04/09,Indian Premier League,T20,Mumbai Indians,Kolkata Knight Riders,Wankhede Stadium,Mumbai,Mumbai Indians,field,N Rana,Mumbai Indians,None,4,20,2.3.0
7,1082598,2017,2017/04/10,Indian Premier League,T20,Kings XI Punjab,Royal Challengers Bangalore,Holkar Cricket Stadium,Indore,Royal Challengers Bangalore,bat,AR Patel,Kings XI Punjab,None,8,20,2.3.0
8,1082599,2017,2017/04/11,Indian Premier League,T20,Rising Pune Supergiant,Delhi Daredevils,Maharashtra Cricket Association Stadium,Pune,Rising Pune Supergiant,field,SV Samson,Delhi Daredevils,97,None,20,2.3.0
9,1082600,2017,2017/04/12,Indian Premier League,T20,Mumbai Indians,Sunrisers Hyderabad,Wankhede Stadium,Mumbai,Mumbai Indians,field,JJ Bumrah,Mumbai Indians,None,4,20,2.3.0


In [11]:
inspection_report = pd.DataFrame(
    [
        {"metric": "delivery_files", "value": len(delivery_files)},
        {"metric": "info_files", "value": len(info_files)},
        {"metric": "missing_info_files", "value": len(missing_info)},
        {"metric": "missing_delivery_files", "value": len(missing_delivery)},
        {"metric": "sample_delivery_rows", "value": len(sample_deliveries)},
        {"metric": "sample_delivery_columns", "value": len(sample_deliveries.columns)},
    ]
)

report_path = DATA_PROCESSED_DIR / "inspection_report.csv"
inspection_report.to_csv(report_path, index=False)

display(inspection_report)
print("Inspection report saved to:", report_path)

,metric,value
0,delivery_files,1243
1,info_files,1243
2,missing_info_files,0
3,missing_delivery_files,0
4,sample_delivery_rows,248
5,sample_delivery_columns,27


Inspection report saved to: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\INDIAN PREMIER LEAGUE\DATA\PROCESSED\inspection_report.csv
